In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.preprocessing import StandardScaler

In [ ]:
AUGMENTATION_FACTOR = 10
EPOCHS = 1000
BATCH_SIZE = 64
SAMPLE_INTERVAL = 50
LATENT_DIM_CATEGORICAL = 150   
LATENT_DIM_COLOR = 200         
LATENT_DIM_CONTINUOUS = 150    
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

In [ ]:
def load_data(csv_path='train.csv'):
    df = pd.read_csv(csv_path, encoding='gbk')
    abundance = df[['abundance']].values.astype(np.float64)
    properties1 = df[['fragment', 'film', 'fiber', 'pellet', 'Foam', 'Others1']].values
    properties2 = df[['<1mm', '1-5mm']].values
    properties3 = df[['PP', 'PET', 'PE', 'PS', 'PA', 'PC', 'PVC', 'Others2']].values
    properties4 = df[['Yellow', 'Red', 'Green', 'Black', 'White', 'Blue', 'Transparent', 'Others3']].values
    original_labels = df['original_risk_label'].values
    feature_groups = {
        'abundance': (abundance, 'continuous'),
        'properties1': (properties1, 'categorical'),
        'properties2': (properties2, 'categorical'),
        'properties3': (properties3, 'categorical'),
        'properties4': (properties4, 'categorical'),
    }
    return feature_groups

In [ ]:
class CategoricalGAN:
    def __init__(self, latent_dim=150):
        self.latent_dim = latent_dim
        self.generator = None
        self.discriminator = None
        self.gan = None

    def build_generator(self, output_dim):
        model = tf.keras.Sequential([
            layers.Dense(256, input_dim=self.latent_dim),
            layers.LeakyReLU(alpha=0.2),
            layers.BatchNormalization(),
            layers.Dense(512),
            layers.LeakyReLU(alpha=0.2),
            layers.BatchNormalization(),
            layers.Dense(512),
            layers.LeakyReLU(alpha=0.2),
            layers.BatchNormalization(),
            layers.Dense(output_dim, activation='softmax')
        ])
        return model

    def build_discriminator(self, input_dim):
        model = tf.keras.Sequential([
            layers.Dense(512, input_dim=input_dim),
            layers.LeakyReLU(alpha=0.2),
            layers.Dropout(0.3),
            layers.Dense(256),
            layers.LeakyReLU(alpha=0.2),
            layers.Dropout(0.3),
            layers.Dense(1, activation='sigmoid')
        ])
        return model

    def train(self, real_data, epochs, batch_size, sample_interval):
        self.output_dim = real_data.shape[1]
        self.generator = self.build_generator(self.output_dim)
        self.discriminator = self.build_discriminator(self.output_dim)

        self.discriminator.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003, beta_1=0.5),
            loss='binary_crossentropy'
        )

        z = layers.Input(shape=(self.latent_dim,))
        generated = self.generator(z)
        self.discriminator.trainable = False
        validity = self.discriminator(generated)
        self.gan = Model(z, validity)
        self.gan.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.5),
            loss='binary_crossentropy'
        )

        valid = np.ones((batch_size, 1)) * 0.9  
        fake = np.zeros((batch_size, 1)) + 0.1

        for epoch in range(epochs):
            idx = np.random.randint(0, real_data.shape[0], batch_size)
            real_samples = real_data[idx]
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            gen_samples = self.generator.predict(noise, verbose=0)

            self.discriminator.trainable = True
            d_loss_real = self.discriminator.train_on_batch(real_samples, valid)
            d_loss_fake = self.discriminator.train_on_batch(gen_samples, fake)
            d_loss = 0.5 * (d_loss_real + d_loss_fake)

            self.discriminator.trainable = False
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            g_loss = self.gan.train_on_batch(noise, valid)

            if epoch % sample_interval == 0:
                noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
                gen_samples = self.generator.predict(noise, verbose=0)
                real_pred = self.discriminator.predict(real_samples, verbose=0)
                fake_pred = self.discriminator.predict(gen_samples, verbose=0)
                print(f"Epoch {epoch} [D loss: {d_loss:.4f}] [G loss: {g_loss:.4f}] "
                      f"[Real prob: {np.mean(real_pred):.4f}, Fake prob: {np.mean(fake_pred):.4f}]")

    def generate_samples(self, n_samples):
        noise = np.random.normal(0, 1, (n_samples, self.latent_dim))
        return self.generator.predict(noise, verbose=0)

In [ ]:
class ContinuousGAN:
    def __init__(self, latent_dim=150):
        self.latent_dim = latent_dim
        self.generator = None
        self.discriminator = None
        self.gan = None
        self.scaler = None

    def build_generator(self, output_dim):
        model = tf.keras.Sequential([
            layers.Dense(256, input_dim=self.latent_dim),
            layers.LeakyReLU(alpha=0.2),
            layers.BatchNormalization(),
            layers.Dense(512),
            layers.LeakyReLU(alpha=0.2),
            layers.BatchNormalization(),
            layers.Dense(512),
            layers.LeakyReLU(alpha=0.2),
            layers.BatchNormalization(),
            layers.Dense(output_dim, activation='linear')
        ])
        return model

    def build_discriminator(self, input_dim):
        model = tf.keras.Sequential([
            layers.Dense(512, input_dim=input_dim),
            layers.LeakyReLU(alpha=0.2),
            layers.Dropout(0.3),
            layers.Dense(256),
            layers.LeakyReLU(alpha=0.2),
            layers.Dropout(0.3),
            layers.Dense(1, activation='sigmoid')
        ])
        return model

    def _preprocess(self, data):
        log_data = np.log1p(data)
        self.scaler = StandardScaler()
        return self.scaler.fit_transform(log_data)

    def _postprocess(self, data):
        inv_scaled = self.scaler.inverse_transform(data)
        original = np.expm1(inv_scaled)
        return np.maximum(original, 0)

    def train(self, real_data, epochs, batch_size, sample_interval):
        self.output_dim = real_data.shape[1]
        processed_data = self._preprocess(real_data)

        self.generator = self.build_generator(self.output_dim)
        self.discriminator = self.build_discriminator(self.output_dim)

        self.discriminator.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.0003, beta_1=0.5),
            loss='binary_crossentropy'
        )

        z = layers.Input(shape=(self.latent_dim,))
        generated = self.generator(z)
        self.discriminator.trainable = False
        validity = self.discriminator(generated)
        self.gan = Model(z, validity)
        self.gan.compile(
            optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001, beta_1=0.5),
            loss='binary_crossentropy'
        )

        valid = np.ones((batch_size, 1)) * 0.9
        fake = np.zeros((batch_size, 1)) + 0.1

        for epoch in range(epochs):
            idx = np.random.randint(0, processed_data.shape[0], batch_size)
            real_samples = processed_data[idx]

            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            gen_samples = self.generator.predict(noise, verbose=0)

            self.discriminator.trainable = True
            d_loss_real = self.discriminator.train_on_batch(real_samples, valid)
            d_loss_fake = self.discriminator.train_on_batch(gen_samples, fake)
            d_loss = 0.5 * (d_loss_real + d_loss_fake)

            self.discriminator.trainable = False
            noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
            g_loss = self.gan.train_on_batch(noise, valid)

            if epoch % sample_interval == 0:
                noise = np.random.normal(0, 1, (batch_size, self.latent_dim))
                gen_samples = self.generator.predict(noise, verbose=0)
                real_pred = self.discriminator.predict(real_samples, verbose=0)
                fake_pred = self.discriminator.predict(gen_samples, verbose=0)
                gen_original = self._postprocess(gen_samples)
                print(f"Epoch {epoch} [D loss: {d_loss:.4f}] [G loss: {g_loss:.4f}] "
                      f"[Real prob: {np.mean(real_pred):.4f}, Fake prob: {np.mean(fake_pred):.4f}] "
                      f"[Gen range: {gen_original.min():.2f} ~ {gen_original.max():.2f}]")

    def generate_samples(self, n_samples):
        noise = np.random.normal(0, 1, (n_samples, self.latent_dim))
        generated = self.generator.predict(noise, verbose=0)
        return self._postprocess(generated)

In [ ]:
def main():
    feature_groups = load_data()

    categorical_configs = [
        ('properties1', ['fragment', 'film', 'fiber', 'pellet', 'Foam', 'Others1'],
         'Shape', LATENT_DIM_CATEGORICAL, 'synthetic_properties_shape'),
        ('properties2', ['<1mm', '1-5mm'],
         'Size', LATENT_DIM_CATEGORICAL, 'synthetic_properties_size'),
        ('properties3', ['PP', 'PET', 'PE', 'PS', 'PA', 'PC', 'PVC', 'Others2'],
         'Type', LATENT_DIM_CATEGORICAL, 'synthetic_properties_type'),
        ('properties4', ['Yellow', 'Red', 'Green', 'Black', 'White', 'Blue', 'Transparent', 'Others3'],
         'Color', LATENT_DIM_COLOR, 'synthetic_properties_color'),
    ]

    for key, columns, desc, latent_dim, out_prefix in categorical_configs:
        data = feature_groups[key][0]
        print(f"\n===== Training {desc} GAN (data dimension: {data.shape[1]}) =====")
        gan = CategoricalGAN(latent_dim=latent_dim)
        gan.train(data, epochs=EPOCHS, batch_size=BATCH_SIZE, sample_interval=SAMPLE_INTERVAL)

        n_synthetic = len(data) * AUGMENTATION_FACTOR
        synthetic = gan.generate_samples(n_synthetic)
        df_out = pd.DataFrame(synthetic, columns=columns)
        out_file = f"{out_prefix}({AUGMENTATION_FACTOR}x).csv"
        df_out.to_csv(out_file, index=False, encoding='utf-8-sig')
        print(f"Saved {out_file}")

    abundance = feature_groups['abundance'][0]
    print(f"\n===== Training Abundance GAN (data dimension: {abundance.shape[1]}) =====")
    cont_gan = ContinuousGAN(latent_dim=LATENT_DIM_CONTINUOUS)
    cont_gan.train(abundance, epochs=EPOCHS, batch_size=BATCH_SIZE, sample_interval=SAMPLE_INTERVAL)

    n_synthetic = len(abundance) * AUGMENTATION_FACTOR
    synthetic_abundance = cont_gan.generate_samples(n_synthetic)
    df_out = pd.DataFrame(synthetic_abundance, columns=['abundance'])
    out_file = f"synthetic_abundance({AUGMENTATION_FACTOR}x).csv"
    df_out.to_csv(out_file, index=False, encoding='utf-8-sig')
    print(f"Saved {out_file}")

    print("\nAll synthetic data generation completed!")